# English to Spanish Translator

## Colab Training Notebook

This notebook trains and evaluates the Transformer model in Google Colab.

Before running the notebook:
- switch Colab to a GPU runtime
- keep `kaggle.json` ready for upload
- review the training settings before starting


## 1. Clone Repository

This cell clones the repository into `/content` and enters the project directory.


In [ ]:
import os

REPO_URL = "https://github.com/mathew-felix/english-spanish-translator.git"
REPO_DIR = "/content/english-spanish-translator"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/english-spanish-translator


## 2. Install Dependencies

Colab already includes PyTorch. This cell installs the remaining packages from `requirements.txt` without replacing the Colab PyTorch build.


In [ ]:
from pathlib import Path

skip_packages = {"torch", "torchvision", "torchaudio"}
filtered_lines = []

for line in Path("requirements.txt").read_text().splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    package_name = stripped.split("==")[0]
    if package_name in skip_packages:
        continue
    filtered_lines.append(stripped)

Path("/tmp/colab-requirements.txt").write_text("\n".join(filtered_lines) + "\n")

!pip install -q -r /tmp/colab-requirements.txt

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Configure Kaggle Credentials

Upload `kaggle.json` from your Kaggle account settings page. The file is stored in `/root/.kaggle/kaggle.json` for this Colab session.


In [ ]:
from google.colab import files
import os
import shutil

kaggle_dir = "/root/.kaggle"
kaggle_path = f"{kaggle_dir}/kaggle.json"

if not os.path.exists(kaggle_path):
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Upload kaggle.json from your Kaggle account settings page.")
    os.makedirs(kaggle_dir, exist_ok=True)
    shutil.move("kaggle.json", kaggle_path)
    os.chmod(kaggle_path, 0o600)

!python -m kaggle.cli --help > /tmp/kaggle-help.txt
print("Kaggle CLI configured.")


## 4. Training Configuration

Adjust these values before running the download, preprocessing, and training cells.

Suggested starting point for a quick Colab check:
- `SAMPLE_FRACTION = 0.02`
- `NUM_EPOCHS = 1`


In [ ]:
SAMPLE_FRACTION = 0.10
NUM_EPOCHS = 10
BATCH_SIZE = 32
MAX_SEQ_LENGTH = 40
LEARNING_RATE = 1e-4
PATIENCE = 3
WARMUP_STEPS = 4000
BEAM_WIDTH = 4


## 5. Download Dataset

This cell downloads `english_spanish.csv` from Kaggle and extracts it into `./data`.


In [ ]:
import os
import subprocess
import sys
import zipfile

DATASET = "djonafegnem/europarl-parallel-corpus-19962011"
FILE_NAME = "english_spanish.csv"
OUTPUT_DIR = "./data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

download_cmd = [
    sys.executable,
    "-m",
    "kaggle.cli",
    "datasets",
    "download",
    "-d",
    DATASET,
    "-f",
    FILE_NAME,
    "-p",
    OUTPUT_DIR,
]

subprocess.run(download_cmd, check=True)

zip_path = os.path.join(OUTPUT_DIR, f"{FILE_NAME}.zip")
with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(OUTPUT_DIR)

print("Downloaded raw dataset to:", os.path.join(OUTPUT_DIR, FILE_NAME))


## 6. Preprocess Dataset

This cell cleans the raw dataset, samples a subset, and writes `train.csv` and `test.csv` into `./data`.


In [ ]:
from source.DatasetPreprocessing import CleanDataset, SmallDataset, Split_data

raw_path = "./data/english_spanish.csv"
cleaned_path = "./data/cleaned_dataset.csv"
sampled_path = "./data/small_dataset.csv"

CleanDataset(raw_path, cleaned_path)
SmallDataset(cleaned_path, sampled_path, SAMPLE_FRACTION)
Split_data(sampled_path)

!ls -lh data


## 7. Train Model

This cell initializes the tokenizer, builds the datasets and dataloaders, trains the Transformer, and reloads the best checkpoint.


In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from source.Config import Config
from source.DatasetTranslation import TranslationDataset
from source.Model import Transformer
from source.Train import train_model, _encode_sentence

config = Config()
config.num_epochs = NUM_EPOCHS
config.batch_size = BATCH_SIZE
config.max_seq_length = MAX_SEQ_LENGTH
config.learning_rate = LEARNING_RATE
config.patience = PATIENCE
config.warmup_steps = WARMUP_STEPS

torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

os.makedirs(config.tokenizer_path, exist_ok=True)

tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
tokenizer.add_special_tokens(
    {
        "additional_special_tokens": [
            config.pad_token,
            config.unk_token,
            config.sos_token,
            config.eos_token,
        ]
    }
)

config.vocab_size = len(tokenizer)
config.pad_token_id = tokenizer.convert_tokens_to_ids(config.pad_token)
config.unk_token_id = tokenizer.convert_tokens_to_ids(config.unk_token)
config.sos_token_id = tokenizer.convert_tokens_to_ids(config.sos_token)
config.eos_token_id = tokenizer.convert_tokens_to_ids(config.eos_token)

tokenizer.save_pretrained(config.tokenizer_path)

train_dataset_full = TranslationDataset(
    csv_file=config.train_csv,
    tokenizer=tokenizer,
    sequence_length=config.max_seq_length,
    config=config,
)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
split_generator = torch.Generator().manual_seed(config.seed)
train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=split_generator,
)

num_workers = 2 if torch.cuda.is_available() else 0
pin_memory = torch.cuda.is_available()

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

model = Transformer(config).to(config.device)
train_model(model, train_dataloader, val_dataloader, config, tokenizer)

checkpoint = torch.load(config.model_save_path, map_location=config.device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Training complete. Best checkpoint:", config.model_save_path)


## 8. Evaluate Model

This cell loads the test split and computes BLEU using the project evaluation code.


In [ ]:
from source.Evaluate import evaluate_model

test_dataset = TranslationDataset(
    csv_file=config.test_csv,
    tokenizer=tokenizer,
    sequence_length=config.max_seq_length,
    config=config,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

avg_bleu = evaluate_model(
    model,
    test_dataloader,
    config,
    tokenizer,
    max_seq_length=config.max_seq_length,
)

print("Final BLEU:", avg_bleu)


## 9. Example Translations

Use the trained checkpoint to translate a few sample English sentences.


In [ ]:
examples = [
    "How are you?",
    "Where is the nearest hospital?",
    "I need help with my homework.",
]

for text in examples:
    encoder_input = _encode_sentence(text, tokenizer, config)
    token_ids = model.generate(encoder_input, config, beam_width=BEAM_WIDTH)
    translation = tokenizer.decode(token_ids, skip_special_tokens=True)
    print(f"{text} -> {translation}")


## 10. Export Artifacts

This cell lists the main training outputs and includes optional download commands for Colab.


In [ ]:
from google.colab import files
import os

for artifact in ["best_model.pth", "loss_plot.png", "bleu_score_distribution.png"]:
    if os.path.exists(artifact):
        print("Ready:", artifact)
    else:
        print("Missing:", artifact)

# files.download("best_model.pth")
# files.download("loss_plot.png")
# files.download("bleu_score_distribution.png")
